# 05 — Gold Fact Table: fact_sales
## Enriched Silver → Star Schema Fact Table

**Purpose:** Build the central fact table of the Gold star schema.
One row per order line item. Contains all measures and foreign keys.
Optimised for analytical querying by BI tools and SQL analysts.

**Input :** `sales_silver.clean_superstore`
**Output:** `sales_gold.fact_sales`
**Layer  :** Gold (fact)
**Grain  :** One row per order line item (order_id + product_id)


## 1. Load config

In [0]:
%run ./00_setup_and_config

In [0]:
log("INFO", "05_gold_fact_sales", "Starting Gold fact table build")


## 2. Read the enriched Silver table

In [0]:
df_silver = spark.table(SILVER_TABLE)

log("INFO", "05_gold_fact_sales", f"Read Silver: {df_silver.count():,} rows")
print(f"  Source rows    : {df_silver.count():,}")
print(f"  Source columns : {len(df_silver.columns)}")
print(f"  Columns        : {df_silver.columns}")


## 3. Understand and document the grain before selecting columns

In [0]:
# GRAIN = the most granular unit of measurement in the fact table.
# Getting the grain wrong is the most common and most costly mistake
# in data modelling. Everything else flows from this decision.
#
# Superstore grain: one row per ORDER LINE ITEM
#   An order can have multiple products → multiple rows per order_id
#   Each row represents: customer X bought product Y in order Z
#
# Verify the grain — if order_id + product_id is unique, our grain is correct.
# If not, we have duplicates at the grain level and need to investigate.

from pyspark.sql import functions as F

total_rows = df_silver.count()
grain_cols = ["order_id", "product_id"]

distinct_grain = df_silver.select(grain_cols).distinct().count()

print("=== GRAIN VERIFICATION ===\n")
print(f"  Total rows              : {total_rows:,}")
print(f"  Distinct order+product  : {distinct_grain:,}")

if total_rows == distinct_grain:
    print(f"  ✅ Grain is unique — one row per order_id + product_id")
    grain_status = "UNIQUE"
else:
    duplicates = total_rows - distinct_grain
    print(f"  ⚠️  {duplicates:,} duplicate grain rows detected")
    print(f"     This means the same product appears twice in one order.")
    print(f"     Investigate before proceeding.")
    grain_status = "DUPLICATE"
    log("WARN", "05_gold_fact_sales", f"Grain not unique: {duplicates} duplicates at order+product level")

log("INFO", "05_gold_fact_sales", f"Grain verification: {grain_status}")


## 4. Add a surrogate key to the fact table

In [0]:
# A surrogate key is a system-generated unique identifier for each row.
# It is NOT the natural business key (order_id) — it is a pipeline-generated key.
#
# Why use a surrogate key?
#   1. Natural keys (order_id) can change if source systems change.
#      Surrogate keys are stable regardless of source changes.
#   2. Integer surrogate keys join faster than long string keys.
#   3. They are the industry standard in dimensional modelling (Kimball methodology).
#
# F.monotonically_increasing_id() generates unique Long integers.
# Note: they are not sequential (gaps exist between partitions) but ARE unique.

df_silver = df_silver.withColumn(
    "fact_id",
    F.monotonically_increasing_id()
)

print(f"  ✅ Surrogate key 'fact_id' added")
print(f"  Type : Long (monotonically increasing, unique per row)")
print(f"  Note : IDs are unique but not sequential across partitions")


## 5. Select exactly the columns needed for fact_sales

In [0]:
# Fact table column categories:
#   1. Surrogate key     — fact_id (system generated)
#   2. Natural keys      — order_id (business identifier)
#   3. Foreign keys      — customer_id, product_id, region (join to dimensions)
#   4. Degenerate dims   — order_date, ship_date, segment, ship_mode
#                          (dimensions too simple to need their own table)
#   5. Date keys         — order_year, order_month, order_quarter
#                          (pre-computed for fast time-series grouping)
#   6. Measures          — sales, quantity, discount, profit (the numbers)
#   7. Derived measures  — profit_margin, shipping_days, discount_amount
#                          (pre-computed KPIs)
#   8. Flags             — is_profitable, sales_band, discount_band
#   9. Audit             — _silver_processed_at (lineage)

fact_columns = [
    # Surrogate key
    "fact_id",

    # Natural business key
    "order_id",

    # Foreign keys → dimension tables
    "customer_id",
    "product_id",
    "region",

    # Degenerate dimensions (no separate dim table needed)
    "order_date",
    "ship_date",
    "segment",
    "ship_mode",
    "city",
    "state",
    "country",
    "postal_code",

    # Pre-computed date parts
    "order_year",
    "order_month",
    "order_quarter",
    "order_yearmonth",
    "order_day_of_week",

    # Core measures
    "sales",
    "quantity",
    "discount",
    "profit",

    # Derived measures
    "profit_margin",
    "shipping_days",
    "shipping_speed",
    "discount_amount",
    "unit_price",

    # Boolean and band flags
    "is_profitable",
    "sales_band",
    "discount_band",

    # Lineage
    "_silver_processed_at",
]

# Verify all expected columns exist in Silver
missing_cols = [c for c in fact_columns if c not in df_silver.columns]
if missing_cols:
    raise Exception(f"Columns missing from Silver: {missing_cols}")

df_fact = df_silver.select(fact_columns)

print("=== FACT TABLE STRUCTURE ===\n")
print(f"  Columns selected : {len(fact_columns)}")
print(f"  Rows             : {df_fact.count():,}")
print()
df_fact.printSchema()

log("INFO", "05_gold_fact_sales", f"Fact columns selected: {len(fact_columns)}")


## 6. Add Gold-layer audit columns

In [0]:
# Each layer adds its own metadata timestamp.
# This gives you a complete lineage trail:
#   _ingested_at         → when Bronze loaded the CSV
#   _silver_processed_at → when Silver cleaned and transformed
#   _gold_written_at     → when Gold fact table was built
#
# Together these timestamps let you calculate pipeline latency
# and identify which step is slowest.

df_fact = df_fact.withColumns({
    "_gold_written_at"  : F.current_timestamp(),
    "_gold_version"     : F.lit("1.0"),
    "_source_table"     : F.lit(SILVER_TABLE),
})

print(f"  ✅ Gold metadata columns added")
print(f"  Total columns in fact_sales : {len(df_fact.columns)}")


## 7. Write fact_sales as a partitioned Delta Table

In [0]:
# PARTITIONING:
#   Partitioning physically organises files on disk by a column's value.
#   When a query filters by order_year, Spark reads ONLY that year's files
#   instead of scanning the entire table.
#
#   partitionBy("order_year", "order_month") means:
#     delta/sales/gold/fact_sales/order_year=2021/order_month=1/part-0000.parquet
#     delta/sales/gold/fact_sales/order_year=2021/order_month=2/part-0000.parquet
#     ...
#
#   A query WHERE order_year = 2023 skips all other years entirely.
#   On a 5-year dataset this means reading 20% of the data instead of 100%.
#   At petabyte scale this is the difference between seconds and hours.

(
    df_fact
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("order_year", "order_month")
    .saveAsTable(FACT_TABLE)
)

log("INFO", "05_gold_fact_sales", f"fact_sales written: {FACT_TABLE}")
print(f"  ✅ {FACT_TABLE} written successfully")
print(f"  Partitioned by : order_year, order_month")


## 8. Comprehensive verification of the written fact table

In [0]:
df_verify = spark.table(FACT_TABLE)
fact_rows = df_verify.count()

print("=== FACT TABLE VERIFICATION ===\n")
print(f"  Rows written         : {fact_rows:,}")
print(f"  Columns              : {len(df_verify.columns)}")

# Verify partitions were created correctly
print("\n  Row distribution by year:\n")
df_verify.groupBy("order_year") \
    .agg(
        F.count("*").alias("row_count"),
        F.round(F.sum("sales"), 2).alias("total_sales"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
    ) \
    .orderBy("order_year") \
    .show()

# Quick measure sanity check
print("  Overall measure totals:\n")
df_verify.select(
    F.round(F.sum("sales"),    2).alias("total_sales"),
    F.round(F.sum("profit"),   2).alias("total_profit"),
    F.round(F.sum("quantity"), 0).alias("total_units"),
    F.round(F.avg("profit_margin"), 4).alias("avg_margin"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.countDistinct("product_id").alias("unique_products"),
).show(truncate=False)

log("INFO", "05_gold_fact_sales", "Fact table verification complete")


## 9. Business queries directly on the fact table

In [0]:
%sql
-- Top 5 performing regions by total sales and profit
SELECT
    region,
    COUNT(*)                        AS transactions,
    ROUND(SUM(sales), 2)            AS total_sales,
    ROUND(SUM(profit), 2)           AS total_profit,
    ROUND(AVG(profit_margin) * 100, 2) AS avg_margin_pct,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) AS overall_margin_pct
FROM sales_gold.fact_sales
GROUP BY region
ORDER BY total_sales DESC

In [0]:
%sql
-- Sales trend by year and quarter
SELECT
    order_year,
    order_quarter,
    COUNT(*)                 AS transactions,
    ROUND(SUM(sales), 2)     AS total_sales,
    ROUND(SUM(profit), 2)    AS total_profit
FROM sales_gold.fact_sales
GROUP BY order_year, order_quarter
ORDER BY order_year, order_quarter

In [0]:
%sql
-- Discount impact on profitability
SELECT
    discount_band,
    COUNT(*)                           AS transactions,
    ROUND(AVG(discount) * 100, 1)      AS avg_discount_pct,
    ROUND(AVG(profit_margin) * 100, 2) AS avg_margin_pct,
    SUM(CASE WHEN is_profitable THEN 1 ELSE 0 END) AS profitable_count,
    ROUND(
        SUM(CASE WHEN is_profitable THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    )                                  AS profitable_pct
FROM sales_gold.fact_sales
GROUP BY discount_band
ORDER BY avg_discount_pct


## 10. Confirm physical partitioning on disk

In [0]:
%sql
DESCRIBE DETAIL sales_gold.fact_sales

In [0]:
# Show the partition structure Spark sees
print("=== PARTITION DISTRIBUTION ===\n")

spark.table(FACT_TABLE) \
    .groupBy("order_year", "order_month") \
    .count() \
    .orderBy("order_year", "order_month") \
    .show(50, truncate=False)

print("""
  ℹ️  Each year/month combination is a physical folder on disk.
     Queries filtering on order_year skip all other partitions entirely.
     This is partition pruning — one of the most impactful optimisations
     available in distributed data processing.
""")

In [0]:
%sql
-- Full transaction log for fact_sales
-- In production, each pipeline run adds a new version entry here
DESCRIBE HISTORY sales_gold.fact_sales


## 11. Notebook completion summary

In [0]:
final_count = spark.table(FACT_TABLE).count()

print("=" * 55)
print("  GOLD FACT TABLE — COMPLETE")
print("=" * 55)
print(f"  Table          : {FACT_TABLE}")
print(f"  Rows           : {final_count:,}")
print(f"  Columns        : {len(df_fact.columns)}")
print(f"  Partitioned by : order_year, order_month")
print(f"  Format         : Delta (ACID, time travel enabled)")
print(f"  Grain          : One row per order line item")
print(f"  Ready for      : dim tables (Notebook 06) + analytics (07)")
print("=" * 55)

log("INFO", "05_gold_fact_sales", "Notebook 05 complete ✅")